In [1]:
from pathlib import Path
import sys, json, hashlib, importlib.util
ROOT=Path("/home/featurize/work/biological_prior_validation").resolve()
INPUT_ZIP=ROOT/"results/summary/task2_step3_review_bundle_task2_ordered_controls_num_v3.zip"
REPORT_NAME="task2_three_comparisons_v1"
TIE_ATOL=1e-12
TIE_RTOL=1e-10  
if not ROOT.is_dir():raise FileNotFoundError(f"Please modify the ROOT：{ROOT}")
if not INPUT_ZIP.is_file():raise FileNotFoundError(f"Missing complete 2.3 verification package：{INPUT_ZIP}")


In [2]:
SOURCE = '"""Task 2.4: immutable score reporting; no count/target reads or model fitting."""\nfrom pathlib import Path\nfrom collections import Counter\nimport json, hashlib, zipfile, io, sys, subprocess\nimport numpy as np\nimport pandas as pd\nENGINE_SHA=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\nMETRICS=[\'mean_poisson_deviance\',\'normalized_mse\']\nMETA=[\'scope_id\',\'cell_line\',\'drug\',\'replicate\',\'n_cells\',\'n_pairs\']\n\ndef sha(path):\n h=hashlib.sha256()\n with Path(path).open(\'rb\') as f:\n  for block in iter(lambda:f.read(1024*1024),b\'\'):h.update(block)\n return h.hexdigest()\ndef json_bytes(obj):return (json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False)+\'\\n\').encode()\ndef write_same(path,data):\n path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n if path.exists() and path.read_bytes()!=data:raise ValueError(\'Different existing output; use a new report version: \'+str(path))\n if not path.exists():path.write_bytes(data)\ndef relative(delta,reference):return float(delta/reference*100) if reference>0 else np.nan\ndef validate_frame(df):\n if len(df)!=1290 or set(df.scope_id)!=set(range(30)) or not df.status.eq(\'OK\').all():raise ValueError(\'Complete all15 OK score table required\')\n if not np.isfinite(df[METRICS].to_numpy(float)).all() or (df[METRICS]<0).any().any():raise ValueError(\'Invalid loss values\')\n for key in [\'scope_id\',\'n_cells\',\'n_pairs\']:\n  v=df[key].to_numpy(float)\n  if not np.isfinite(v).all() or (v!=np.floor(v)).any() or (v<0).any():raise ValueError(\'Invalid integer identity/count: \'+key)\n seed=df.loc[df.method.isin([\'M3\',\'M4\']),\'seed_index\'].to_numpy(float)\n if not np.isfinite(seed).all() or (seed!=np.floor(seed)).any():raise ValueError(\'Invalid seed indices\')\n if df.loc[df.method.isin([\'M0\',\'M1\',\'M2\']),\'seed_index\'].notna().any():raise ValueError(\'Unexpected seed for deterministic method\')\n for sid,g in df.groupby(\'scope_id\'):\n  if g.method.value_counts().to_dict()!={\'M3\':20,\'M4\':20,\'M0\':1,\'M1\':1,\'M2\':1}:raise ValueError(\'Method/seed roster mismatch\')\n  for m in [\'M3\',\'M4\']:\n   if sorted(g.loc[g.method.eq(m),\'seed_index\'].astype(int))!=list(range(20)):raise ValueError(\'Incomplete seed indices\')\n  for c in META[1:]+[\'gene_sha256\',\'Lc_sha256\',\'common_I_row_sha256\']:\n   if g[c].nunique(dropna=False)!=1 or g[c].isna().any():raise ValueError(\'Common-I metadata mismatch: \'+c)\n  r=g.loc[g.method.eq(\'M2\')].iloc[0];m3=g.loc[g.method.eq(\'M3\')]\n  for c in [\'alpha\',\'beta\',\'trace_Lg\',\'weight_sum\',\'zero_weight_edges\']:\n   if not np.allclose(m3[c].to_numpy(float),float(r[c]),rtol=1e-10,atol=1e-12):raise ValueError(\'Matched control not matched: \'+c)\n  if not m3.lambda_rule.eq(r.lambda_rule).all():raise ValueError(\'Matched parameters mismatch\')\n\ndef read_verified_bundle(path):\n with zipfile.ZipFile(path) as z:\n  if len(set(z.namelist()))!=len(z.namelist()) or z.testzip() is not None:raise ValueError(\'Corrupt/duplicate ZIP entries\')\n  js=lambda n:json.loads(z.read(n))\n  manifest=js(\'review_manifest.json\')\n  for item in manifest[\'files\']:\n   if hashlib.sha256(z.read(item[\'name\'])).hexdigest()!=item[\'sha256\']:raise ValueError(\'Manifest mismatch: \'+item[\'name\'])\n  contract=js(\'experiment_contract.json\');effective=js(\'effective_parameter_grid.json\');summary=js(\'task2_step3_summary.json\')\n  if not summary[\'full15_run\'] or js(\'all15_summary.json\')[\'status\']!=\'all15_exploratory_completed\':raise ValueError(\'All15 run incomplete\')\n  if summary[\'old_lock_before\']!=summary[\'old_lock_after\']:raise ValueError(\'Old target lock changed\')\n  if hashlib.sha256(z.read(\'src/task2_experiments.py\')).hexdigest()!=contract[\'engine_sha256\']:raise ValueError(\'Fitting engine mismatch\')\n  if hashlib.sha256(json.dumps(effective,sort_keys=True).encode()).hexdigest()!=contract[\'effective_grid_sha256\']:raise ValueError(\'Effective grid mismatch\')\n  df=pd.read_csv(io.BytesIO(z.read(\'all15_all_scores.csv\')));validate_frame(df)\n  genes=pd.read_csv(io.BytesIO(z.read(\'outer_gene_ids.csv\')))\n  if not np.allclose(df.n_pairs,df.n_cells*len(genes)):raise ValueError(\'Incomplete pair coverage\')\n  for col,total in [(\'mean_poisson_deviance\',\'deviance_sum\'),(\'normalized_mse\',\'normalized_sse_sum\')]:\n   if not np.allclose(df[col],df[total]/df.n_pairs,rtol=1e-10,atol=1e-12):raise ValueError(\'Score/sufficient-sum mismatch\')\n  for s in contract[\'all_scopes\']:\n   g=df[df.scope_id.eq(s[\'scope_id\'])]\n   if any(not g[c].eq(s[c]).all() for c in META):raise ValueError(\'Contract scope mismatch\')\n   p=f"scopes/scope_{s[\'scope_id\']:02d}/precision/";gate=js(p+\'gate.json\');probes=pd.read_csv(io.BytesIO(z.read(p+\'probes.csv\')))\n   if gate[\'status\']!=\'PASS\' or gate[\'final_B_used\'] is not False or len(probes)!=19 or not probes.status.eq(\'PASS\').all():raise ValueError(\'Real precision gate failed\')\n   for key in [\'relative_prediction_l2\',\'relative_development_deviance\',\'absolute_s_max\']:\n    values=probes.loc[probes.method.eq(\'M2\'),key] if key==\'absolute_s_max\' else probes[key]\n    if not np.isfinite(values).all() or not values.le(effective[\'precision\'][\'thresholds\'][key]).all():raise ValueError(\'Precision values exceed threshold\')\n  for stage in [\'fixed\',\'adaptive\',\'matched\',\'rewired\']:\n   prefix=f\'scores/all15/{stage}/\';ledger=js(prefix+\'target_access_ledger.json\');registry=js(prefix+\'frozen_registry.json\')\n   if ledger[\'registry_sha256\']!=hashlib.sha256(z.read(prefix+\'frozen_registry.json\')).hexdigest() or registry[\'experiment_contract_sha256\']!=hashlib.sha256(z.read(\'experiment_contract.json\')).hexdigest():raise ValueError(\'Registry/ledger linkage mismatch\')\n   expected=[(r[\'scope\'][\'scope_id\'],r[\'method\'],None if r[\'seed_index\'] is None else int(r[\'seed_index\'])) for r in registry[\'records\']]\n   stage_df=pd.read_csv(io.BytesIO(z.read(prefix+\'scores.csv\')))\n   actual=[(r.scope_id,r.method,None if pd.isna(r.seed_index) else int(r.seed_index)) for r in stage_df.itertuples()]\n   if Counter(expected)!=Counter(actual):raise ValueError(\'Stage registry/score coverage mismatch\')\n   compare=df[df.method.isin(stage_df.method.unique())].set_index([\'scope_id\',\'method\',\'seed_index\']).sort_index()\n   stage_df=stage_df.set_index([\'scope_id\',\'method\',\'seed_index\']).sort_index()\n   if not np.allclose(compare[METRICS],stage_df[METRICS],rtol=1e-12,atol=1e-14):raise ValueError(\'Stage scores differ from summary\')\n  failures=[];tuning_counts=Counter();edge_counts={}\n  for n in z.namelist():\n   if \'/inner/tuning/\' in n and n.endswith(\'search_table.csv\'):\n    t=pd.read_csv(io.BytesIO(z.read(n)));method=n.split(\'/inner/tuning/\')[1].split(\'/\')[0].split(\'_\')[0]\n    for status,count in t.status.value_counts().items():tuning_counts[(method,status)]+=int(count)\n    for row in t.loc[~t.status.eq(\'OK\')].to_dict(\'records\'):failures.append({\'source_table\':n,\'method\':method,**row})\n  for sid in df.scope_id.unique():\n   a=js(f\'scopes/scope_{sid:02d}/outer/models/M2/graph_audit.json\');edge_counts[int(sid)]=int(a[\'effective_gene\'][\'support_edges\'])\n  return df,contract,effective,pd.DataFrame(failures),pd.DataFrame([{\'method\':m,\'status\':s,\'n_candidates\':n} for (m,s),n in sorted(tuning_counts.items())]),edge_counts\n\ndef calculate_tables(df,tie_atol=1e-12,tie_rtol=1e-10):\n validate_frame(df)\n if not np.isfinite([tie_atol,tie_rtol]).all() or min(tie_atol,tie_rtol)<0:raise ValueError(\'Invalid tie tolerance\')\n fixed=[];pairs=[];matched=[];null_pairs=[];nulls=[]\n for sid,g in df.groupby(\'scope_id\',sort=True):\n  meta=g[META].iloc[0].to_dict();base={m:g[g.method.eq(m)].iloc[0] for m in [\'M0\',\'M1\',\'M2\']}\n  for metric in METRICS:\n   for label,method,ref in [(\'fixed\',\'M1\',\'M0\'),(\'adaptive\',\'M2\',\'M1\')]:\n    obs=float(base[method][metric]);reference=float(base[ref][metric]);delta=obs-reference\n    fixed.append({**meta,\'metric\':metric,\'comparison\':label,\'method\':method,\'reference_method\':ref,\'score\':obs,\'reference_score\':reference,\'delta_absolute\':delta,\'delta_percent\':relative(delta,reference),\'relative_available\':reference>0})\n   values=[];percents=[]\n   for r in g[g.method.eq(\'M3\')].sort_values(\'seed_index\').to_dict(\'records\'):\n    reference=float(r[metric]);obs=float(base[\'M2\'][metric]);delta=obs-reference;percent=relative(delta,reference);values.append(delta);percents.append(percent)\n    pairs.append({**meta,\'metric\':metric,\'seed_index\':int(r[\'seed_index\']),\'M2_score\':obs,\'M3_score\':reference,\'delta_absolute\':delta,\'delta_percent\':percent,\'relative_available\':reference>0,\'M2_conditional_EDF\':base[\'M2\'][\'conditional_trace_H\'],\'M3_conditional_EDF\':r[\'conditional_trace_H\'],\'conditional_EDF_delta_M2_minus_M3\':base[\'M2\'][\'conditional_trace_H\']-r[\'conditional_trace_H\']})\n   refs=g[g.method.eq(\'M3\')][metric].to_numpy(float);ref_mean=float(refs.mean());obs=float(base[\'M2\'][metric]);delta=obs-ref_mean\n   matched.append({**meta,\'metric\':metric,\'comparison\':\'matched\',\'n_seeds\':len(refs),\'M2_score\':obs,\'M3_mean\':ref_mean,\'M3_median\':float(np.median(refs)),\'M3_q05\':float(np.quantile(refs,.05)),\'M3_q95\':float(np.quantile(refs,.95)),\'delta_absolute_vs_mean\':delta,\'delta_percent_vs_mean\':relative(delta,ref_mean),\'mean_seed_delta_percent\':float(np.nanmean(percents)) if np.isfinite(percents).any() else np.nan,\'delta_seed_q05\':float(np.quantile(values,.05)),\'delta_seed_q95\':float(np.quantile(values,.95)),\'n_M2_better\':int((np.array(values)<0).sum()),\'n_equal_exact\':int((np.array(values)==0).sum()),\'n_M2_worse\':int((np.array(values)>0).sum()),\'relative_available\':ref_mean>0})\n   refs=g[g.method.eq(\'M4\')][metric].to_numpy(float);obs=float(base[\'M1\'][metric]);tol=tie_atol+tie_rtol*abs(obs)\n   lower=int((refs<obs-tol).sum());ties=int((np.abs(refs-obs)<=tol).sum());higher=int((refs>obs+tol).sum());assert lower+ties+higher==len(refs)\n   for r in g[g.method.eq(\'M4\')].sort_values(\'seed_index\').to_dict(\'records\'):\n    ref=float(r[metric]);d=obs-ref;null_pairs.append({**meta,\'metric\':metric,\'seed_index\':int(r[\'seed_index\']),\'M1_score\':obs,\'M4_score\':ref,\'delta_absolute_M1_minus_M4\':d,\'delta_percent_M1_minus_M4\':relative(d,ref),\'relative_available\':ref>0})\n   ref_mean=float(refs.mean());d=obs-ref_mean\n   nulls.append({**meta,\'metric\':metric,\'n_seeds\':len(refs),\'M1_score\':obs,\'M4_mean\':ref_mean,\'M4_median\':float(np.median(refs)),\'M4_q05\':float(np.quantile(refs,.05)),\'M4_q95\':float(np.quantile(refs,.95)),\'delta_absolute_M1_minus_null_mean\':d,\'delta_percent_M1_minus_null_mean\':relative(d,ref_mean),\'n_null_better\':lower,\'n_numerical_ties\':ties,\'n_null_worse\':higher,\'M1_rank_best\':1+lower,\'M1_rank_worst\':1+lower+ties,\'M1_midrank_percentile\':100*(lower+.5*ties)/len(refs),\'tie_score_tolerance\':tol,\'relative_available\':ref_mean>0})\n return {\'fixed_adaptive_effects\':pd.DataFrame(fixed),\'matched_seed_effects\':pd.DataFrame(pairs),\'matched_summary\':pd.DataFrame(matched),\'M4_seed_effects\':pd.DataFrame(null_pairs),\'M4_null_positions\':pd.DataFrame(nulls)}\n\ndef consistency(tables):\n f=tables[\'fixed_adaptive_effects\'].rename(columns={\'delta_absolute\':\'delta\',\'delta_percent\':\'percent\'})\n m=tables[\'matched_summary\'].rename(columns={\'delta_absolute_vs_mean\':\'delta\',\'delta_percent_vs_mean\':\'percent\'})\n records=[]\n for key,g in pd.concat([f,m],ignore_index=True).groupby([\'cell_line\',\'drug\',\'metric\',\'comparison\']):\n  g=g.sort_values(\'replicate\')\n  if len(g)!=2:raise ValueError(\'Exactly two replicate labels per context required\')\n  v=g.delta.to_numpy(float);label=\'both_better\' if (v<0).all() else \'both_worse\' if (v>0).all() else \'both_exact_zero\' if (v==0).all() else \'opposite_or_zero\'\n  records.append(dict(zip([\'cell_line\',\'drug\',\'metric\',\'comparison\'],key),replicate_label_1=g.iloc[0].replicate,replicate_label_2=g.iloc[1].replicate,delta_label_1=v[0],delta_label_2=v[1],percent_label_1=g.iloc[0].percent,percent_label_2=g.iloc[1].percent,sign_consistency=label,interpretation=\'descriptive label consistency, not biological cross-replicate directions\'))\n return pd.DataFrame(records)\n\ndef run_report(input_zip,root,report_name=\'task2_three_comparisons_v1\',tie_atol=1e-12,tie_rtol=1e-10,self_tests=None):\n root=Path(root).resolve();input_zip=Path(input_zip).resolve()\n if Path(report_name).name!=report_name or report_name in [\'\',\'.\',\'..\']:raise ValueError(\'Invalid report name\')\n if self_tests is not None and self_tests.get(\'status\')!=\'PASS\':raise ValueError(\'Self tests must pass\')\n df,source_contract,effective,failed,candidates,edge_counts=read_verified_bundle(input_zip)\n try:git_commit=subprocess.run([\'git\',\'rev-parse\',\'--verify\',\'HEAD\'],cwd=root,capture_output=True,text=True,check=True).stdout.strip()\n except (OSError,subprocess.CalledProcessError):git_commit=None\n config={\'schema\':\'task2-step4-report-v1\',\'input_zip_sha256\':sha(input_zip),\'source_experiment_engine_sha256\':source_contract[\'engine_sha256\'],\'report_engine_sha256\':ENGINE_SHA,\'metrics\':METRICS,\'sign\':\'negative delta means lower loss\',\'formulas\':{\'fixed\':\'D(M1)-D(M0), percentage denominator D(M0)\',\'adaptive\':\'D(M2)-D(M1), denominator D(M1)\',\'matched_seed\':\'D(M2)-D(M3_seed), denominator D(M3_seed)\',\'matched_scope\':\'D(M2)-mean_seed D(M3), denominator mean_seed D(M3)\',\'M4\':\'D(M1)-D(M4_seed), denominator D(M4_seed)\'},\'M3_summary_percent\':\'ratio against mean null loss; distinct from mean of per-seed ratios\',\'tie_atol\':tie_atol,\'tie_rtol\':tie_rtol,\'seed_quantiles\':\'5/95 percentiles of 20 saved randomized runs, descriptive ranges, NOT biological confidence intervals\',\'inference\':\'No formal p-values/multiple-testing claims or cell bootstrap. Independent biological sampling units unverified; biological uncertainty not estimable from this bundle. All contexts retained. Finite graph swaps not proven uniform/mixed.\',\'replicates\':\'Scenario A label-stratified molecular holdout; Scenario B blocked\',\'target_access\':\'Reads saved score/audit ZIP only; no raw B access, fitting, tuning or seed extension\',\'git_commit\':git_commit,\'environment\':{\'python\':sys.version,\'numpy\':np.__version__,\'pandas\':pd.__version__}}\n config_dir=root/\'configs\'/report_name;out=root/\'results/summary\'/report_name;raw=root/\'results/raw\'/report_name;report_dir=root/\'reports\'/report_name\n write_same(config_dir/\'report_contract.json\',json_bytes(config))\n tables=calculate_tables(df,tie_atol,tie_rtol);tables[\'label_sign_consistency\']=consistency(tables)\n tables[\'failed_candidates\']=failed;tables[\'candidate_status_summary\']=candidates\n audit=[]\n for sid,g in df.groupby(\'scope_id\'):\n  r=g[g.method.eq(\'M2\')].iloc[0];m3=g[g.method.eq(\'M3\')];audit.append({**r[META].to_dict(),\'support_edges\':edge_counts[sid],\'M2_zero_edge_fraction\':float(r.zero_weight_edges/edge_counts[sid]) if edge_counts[sid]>0 else np.nan,\'M2_weight_sum\':r.weight_sum,\'M2_trace_Lg\':r.trace_Lg,\'M2_conditional_EDF\':r.conditional_trace_H,\'M3_conditional_EDF_mean\':float(m3.conditional_trace_H.mean()),\'M3_EDF_relative_percent_mean\':float(m3.conditional_trace_H_relative_percent.mean()),\'note\':\'Matched total weight and trace do not imply matched conditional EDF; conditional EDF is not total adaptive DF\'})\n tables[\'smoothing_audit\']=pd.DataFrame(audit)\n write_same(raw/\'all15_complete_benchmark.csv\',df.to_csv(index=False).encode())\n for name,table in tables.items():write_same(out/(name+\'.csv\'),table.to_csv(index=False).encode())\n summary={\'status\':\'COMPLETE\',\'n_contexts\':15,\'n_scope_labels\':30,\'n_score_rows\':len(df),\'n_formal_precision_probes\':570,\'n_failed_candidates\':len(failed),\'new_raw_B_access\':False,\'scenario_B\':\'BLOCKED\',\'source_input_sha256\':config[\'input_zip_sha256\'],\'old_target_lock\':source_contract[\'old_target_history\'][\'target_access_lock_sha256\']}\n write_same(out/\'summary.json\',json_bytes(summary))\n memo=\'任务2.4：三组主要对比\\n\\n负差值表示改善。所有15 contexts和两个replicate labels完整保留。\\nfixed=M1-M0；adaptive=M2-M1；matched=M2-M3。matched逐seed表和null均值摘要同时报告，两种相对百分比聚合不可混淆。M4单独报告M1在随机图分布中的排名和分位数，lower percentile对应更低loss。\\nseed分位数是技术随机化描述，非biological置信区间；不提供正式p值或显著性结论。独立sample/plate/well来源未确认，无法由cell数估计biological不确定性。M3的conditional EDF变化单独保留，不能把matched差异直接归因于biological edge identity。\\n全部失败候选保留。当前报告只读取已冻结的scores/audits，不读取原始B、不重拟合、不重新选参数或seed。重要结果的100-seed扩展属于后续新版本工作。\\n\'\n write_same(report_dir/\'interpretation_notes.txt\',memo.encode())\n paths=[config_dir/\'report_contract.json\',raw/\'all15_complete_benchmark.csv\',report_dir/\'interpretation_notes.txt\']+sorted(out.glob(\'*.csv\'))+[out/\'summary.json\']\n if self_tests is not None:\n  write_same(config_dir/\'self_tests.json\',json_bytes(self_tests));paths.append(config_dir/\'self_tests.json\')\n manifest={\'files\':[{\'name\':str(p.relative_to(root)),\'sha256\':sha(p)} for p in paths]};write_same(out/\'report_manifest.json\',json_bytes(manifest));paths.append(out/\'report_manifest.json\')\n bundle=root/\'results/summary\'/f\'{report_name}_review_bundle.zip\'\n with zipfile.ZipFile(bundle,\'w\',zipfile.ZIP_DEFLATED) as z:\n  for p in paths:z.write(p,str(p.relative_to(root)))\n  z.write(Path(__file__),\'src/task2_reporting.py\')\n return tables,summary,bundle\n'
path=ROOT/"src/task2_reporting.py"
path.parent.mkdir(parents=True,exist_ok=True)
if path.exists() and path.read_text(encoding="utf-8")!=SOURCE:
    digest=hashlib.sha256(path.read_bytes()).hexdigest()[:12]
    backup=path.with_name("task2_reporting."+digest+".backup.py")
    if not backup.exists():backup.write_bytes(path.read_bytes())
path.write_text(SOURCE,encoding="utf-8")
spec=importlib.util.spec_from_file_location("task2_reporting",path)
rp=importlib.util.module_from_spec(spec)
sys.modules["task2_reporting"]=rp
spec.loader.exec_module(rp)
print("Reporting engine:",rp.ENGINE_SHA)


Reporting engine: f04a28bef4550d6a7e99f3534ce3656575d8707c1fb9adcab0f8952354eab922


In [3]:
TEST_SOURCE = "import numpy as np\nimport pandas as pd\nimport task2_reporting as rp\n\ndef fixture():\n rows=[]\n for sid in range(30):\n  for m in ['M0','M1','M2','M3','M4']:\n   seeds=range(20) if m in ['M3','M4'] else [None]\n   for seed in seeds:\n    value={'M0':1.,'M1':.9,'M2':.8,'M3':.85,'M4':.9}[m]\n    row={'scope_id':sid,'cell_line':['A','B','C'][sid//10],'drug':str((sid//2)%5),'replicate':str(sid%2),'n_cells':2,'n_pairs':4,'method':m,'seed_index':seed,'status':'OK','mean_poisson_deviance':value,'normalized_mse':value,'gene_sha256':'genes','Lc_sha256':'Lc','common_I_row_sha256':'rows','alpha':1.,'beta':1.,'lambda_rule':'q0.5' if m in ['M2','M3'] else 'fixed_limit','trace_Lg':10.,'weight_sum':5.,'zero_weight_edges':2,'conditional_trace_H':80. if m=='M2' else 81.}\n    rows.append(row)\n return pd.DataFrame(rows)\ndef rejected(fn):\n try:fn()\n except ValueError:return\n raise AssertionError('Expected rejection')\ndef run_tests():\n df=fixture();before=df.copy(deep=True);t=rp.calculate_tables(df)\n assert len(t['fixed_adaptive_effects'])==120 and len(t['matched_seed_effects'])==1200 and len(t['matched_summary'])==60 and len(t['M4_null_positions'])==60\n f=t['fixed_adaptive_effects'];assert np.allclose(f.delta_absolute,-.1)\n assert np.allclose(f.loc[f.comparison.eq('fixed'),'delta_percent'],-10.)\n assert np.allclose(f.loc[f.comparison.eq('adaptive'),'delta_percent'],-100/9)\n matched=t['matched_summary'];assert np.allclose(matched.delta_absolute_vs_mean,-.05) and np.allclose(matched.delta_percent_vs_mean,-100/17)\n null=t['M4_null_positions'];assert null.n_numerical_ties.eq(20).all() and null.M1_midrank_percentile.eq(50).all() and null.M1_rank_best.eq(1).all() and null.M1_rank_worst.eq(21).all()\n c=rp.consistency(t);assert len(c)==90 and c.sign_consistency.eq('both_better').all()\n pd.testing.assert_frame_equal(before,df)\n for column in rp.METRICS:\n  zero=df.copy();zero[column]=0.;z=rp.calculate_tables(zero);assert z['fixed_adaptive_effects'].loc[z['fixed_adaptive_effects'].metric.eq(column),'delta_percent'].isna().all()\n  assert z['M4_null_positions'].loc[z['M4_null_positions'].metric.eq(column),'M1_midrank_percentile'].eq(50).all()\n shifted=df.copy();shifted.loc[shifted.method.eq('M4'),rp.METRICS]=.95;n=rp.calculate_tables(shifted)['M4_null_positions'];assert n.n_null_worse.eq(20).all() and n.M1_midrank_percentile.eq(0).all()\n shifted.loc[shifted.method.eq('M4'),rp.METRICS]=.85;n=rp.calculate_tables(shifted)['M4_null_positions'];assert n.n_null_better.eq(20).all() and n.M1_midrank_percentile.eq(100).all()\n random=df.copy();mask=random.method.eq('M3');random.loc[mask,'mean_poisson_deviance']=.81+random.loc[mask,'seed_index']*.01\n a=rp.calculate_tables(random)['matched_summary'];primary=a[a.metric.eq('mean_poisson_deviance')]\n assert not np.allclose(primary.mean_seed_delta_percent,primary.delta_percent_vs_mean)\n rejected(lambda:rp.validate_frame(df.iloc[:-1]))\n bad=df.copy();bad.loc[bad.method.eq('M3'),'seed_index']=0;rejected(lambda:rp.validate_frame(bad))\n bad=df.copy();bad.loc[bad.method.eq('M4'),'seed_index']+=.1;rejected(lambda:rp.validate_frame(bad))\n for key,value in [('status','FAILED'),('mean_poisson_deviance',np.inf),('common_I_row_sha256','changed'),('beta',2.)]:\n  bad=df.copy();bad.loc[bad.method.eq('M3') & bad.seed_index.eq(0),key]=value;rejected(lambda:rp.validate_frame(bad))\n rejected(lambda:rp.calculate_tables(df,tie_rtol=-1))\n return {'status':'PASS','checks':['PDF subtraction signs and baseline denominators','All 15 contexts and seed rows retained; input unchanged','Zero reference remains NA, all-tie/better/worse null ranks','Mean per-seed ratio differs from ratio against mean null','Label consistency remains descriptive','Incomplete/duplicate/fractional seed, nonfinite loss, unmatched parameters/common-I rejected']}\nif __name__=='__main__':\n import json\n print(json.dumps(run_tests(),indent=2))\n"
ns={"__name__":"task2_reporting_tests"}
exec(compile(TEST_SOURCE,"embedded_task2_reporting_tests.py","exec"),ns)
TEST_REPORT=ns["run_tests"]()
assert TEST_REPORT["status"]=="PASS"
print(json.dumps(TEST_REPORT,ensure_ascii=False,indent=2))


{
  "status": "PASS",
  "checks": [
    "PDF subtraction signs and baseline denominators",
    "All 15 contexts and seed rows retained; input unchanged",
    "Zero reference remains NA, all-tie/better/worse null ranks",
    "Mean per-seed ratio differs from ratio against mean null",
    "Label consistency remains descriptive",
    "Incomplete/duplicate/fractional seed, nonfinite loss, unmatched parameters/common-I rejected"
  ]
}


In [4]:
TABLES,SUMMARY,REVIEW_ZIP=rp.run_report(
    INPUT_ZIP,ROOT,REPORT_NAME,
    tie_atol=TIE_ATOL,tie_rtol=TIE_RTOL,self_tests=TEST_REPORT)
print(json.dumps(SUMMARY,ensure_ascii=False,indent=2))
print("Verification package:",REVIEW_ZIP)


{
  "status": "COMPLETE",
  "n_contexts": 15,
  "n_scope_labels": 30,
  "n_score_rows": 1290,
  "n_formal_precision_probes": 570,
  "n_failed_candidates": 8,
  "new_raw_B_access": false,
  "scenario_B": "BLOCKED",
  "source_input_sha256": "d72cb4f674c7a65b898ebecf5ea012b009054b805be9079ca73ecd2580aafb45",
  "old_target_lock": "c166f83b5ff000156fb56c7f6c39084086f89d439fd5fd9396a3735d9bd6d9fe"
}
Verification package: /home/featurize/work/biological_prior_validation/results/summary/task2_three_comparisons_v1_review_bundle.zip


In [5]:
PRIMARY="mean_poisson_deviance"
EFFECTS=TABLES["fixed_adaptive_effects"]
cols=["cell_line","drug","replicate","comparison","score","reference_score","delta_absolute","delta_percent"]
print(EFFECTS.loc[EFFECTS.metric.eq(PRIMARY),cols].to_string(index=False))


cell_line       drug   replicate comparison    score  reference_score  delta_absolute  delta_percent
     A172    Vehicle replicate_1      fixed 0.916029         0.916029    3.040669e-08       0.000003
     A172    Vehicle replicate_1   adaptive 0.916020         0.916029   -9.092641e-06      -0.000993
     A172    Vehicle replicate_2      fixed 0.892129         0.892129    0.000000e+00       0.000000
     A172    Vehicle replicate_2   adaptive 0.892122         0.892129   -7.011630e-06      -0.000786
     A172  Lapatinib replicate_1      fixed 0.914980         0.914980    2.572293e-08       0.000003
     A172  Lapatinib replicate_1   adaptive 0.914969         0.914980   -1.117165e-05      -0.001221
     A172  Lapatinib replicate_2      fixed 0.898025         0.898025    0.000000e+00       0.000000
     A172  Lapatinib replicate_2   adaptive 0.898000         0.898025   -2.553965e-05      -0.002844
     A172 Nintedanib replicate_1      fixed 0.855716         0.855706    1.010999e-05      

In [6]:
MATCHED=TABLES["matched_summary"]
cols=["cell_line","drug","replicate","M2_score","M3_mean","delta_absolute_vs_mean","delta_percent_vs_mean","mean_seed_delta_percent","delta_seed_q05","delta_seed_q95","n_M2_better","n_M2_worse"]
print(MATCHED.loc[MATCHED.metric.eq(PRIMARY),cols].to_string(index=False))
print("Complete per-seed row count:",len(TABLES["matched_seed_effects"]))


cell_line       drug   replicate  M2_score  M3_mean  delta_absolute_vs_mean  delta_percent_vs_mean  mean_seed_delta_percent  delta_seed_q05  delta_seed_q95  n_M2_better  n_M2_worse
     A172    Vehicle replicate_1  0.916020 0.918823               -0.002803              -0.305071                -0.305062       -0.003357       -0.002305           20           0
     A172    Vehicle replicate_2  0.892122 0.898366               -0.006244              -0.695057                -0.695005       -0.007072       -0.004971           20           0
     A172  Lapatinib replicate_1  0.914969 0.918117               -0.003148              -0.342885                -0.342859       -0.003890       -0.002546           20           0
     A172  Lapatinib replicate_2  0.898000 0.903955               -0.005955              -0.658813                -0.658760       -0.006833       -0.004937           20           0
     A172 Nintedanib replicate_1  0.855697 0.859467               -0.003770              -0.438

In [7]:
NULLS=TABLES["M4_null_positions"]
cols=["cell_line","drug","replicate","M1_score","M4_mean","M4_q05","M4_q95","delta_absolute_M1_minus_null_mean","delta_percent_M1_minus_null_mean","n_null_better","n_numerical_ties","n_null_worse","M1_rank_best","M1_rank_worst","M1_midrank_percentile"]
print(NULLS.loc[NULLS.metric.eq(PRIMARY),cols].to_string(index=False))


cell_line       drug   replicate  M1_score  M4_mean   M4_q05   M4_q95  delta_absolute_M1_minus_null_mean  delta_percent_M1_minus_null_mean  n_null_better  n_numerical_ties  n_null_worse  M1_rank_best  M1_rank_worst  M1_midrank_percentile
     A172    Vehicle replicate_1  0.916029 0.916029 0.916029 0.916029                      -1.002000e-08                     -1.093851e-06              0                 0            20             1              1                    0.0
     A172    Vehicle replicate_2  0.892129 0.892129 0.892129 0.892129                       2.220446e-16                      2.488931e-14              0                20             0             1             21                   50.0
     A172  Lapatinib replicate_1  0.914980 0.914980 0.914980 0.914980                      -9.247607e-09                     -1.010689e-06              0                 0            20             1              1                    0.0
     A172  Lapatinib replicate_2  0.898025 0.898

In [8]:
CONSISTENCY=TABLES["label_sign_consistency"]
print(CONSISTENCY.loc[CONSISTENCY.metric.eq(PRIMARY)].to_string(index=False))
print("\nConditional smoothing audit：")
print(TABLES["smoothing_audit"].to_string(index=False))


cell_line       drug                metric comparison replicate_label_1 replicate_label_2  delta_label_1  delta_label_2  percent_label_1  percent_label_2 sign_consistency                                                           interpretation
     A172  Lapatinib mean_poisson_deviance   adaptive       replicate_1       replicate_2  -1.117165e-05  -2.553965e-05        -0.001221        -0.002844      both_better descriptive label consistency, not biological cross-replicate directions
     A172  Lapatinib mean_poisson_deviance      fixed       replicate_1       replicate_2   2.572293e-08   0.000000e+00         0.000003         0.000000 opposite_or_zero descriptive label consistency, not biological cross-replicate directions
     A172  Lapatinib mean_poisson_deviance    matched       replicate_1       replicate_2  -3.148085e-03  -5.955376e-03        -0.342885        -0.658813      both_better descriptive label consistency, not biological cross-replicate directions
     A172 Nintedanib mea

In [9]:
print(TABLES["candidate_status_summary"].to_string(index=False))
print("\nFailed candidate:")
FAILED=TABLES["failed_candidates"]
if FAILED.empty:
    print("None")
else:
    cols=[c for c in ["source_table","method","alpha","beta","lambda_rule","status","failure_reason"] if c in FAILED]
    print(FAILED[cols].to_string(index=False))
print("\nPlease return：",REVIEW_ZIP)
print("Results Table:",ROOT/"results/summary"/REPORT_NAME)
print("Complete 1290-line benchmark:",ROOT/"results/raw"/REPORT_NAME/"all15_complete_benchmark.csv")


method status  n_candidates
    M0     OK           390
    M1     OK          3180
    M2 FAILED             8
    M2     OK         19912
    M4     OK         63600

Failed candidate:
                                    source_table method     alpha       beta     lambda_rule status                      failure_reason
scopes/scope_02/inner/tuning/M2/search_table.csv     M2 100.00000 100.000000           q0.25 FAILED Adaptive maximum iterations reached
scopes/scope_08/inner/tuning/M2/search_table.csv     M2   0.00000 100.000000            q0.5 FAILED Adaptive maximum iterations reached
scopes/scope_08/inner/tuning/M2/search_table.csv     M2   0.00100 100.000000            q0.5 FAILED Adaptive maximum iterations reached
scopes/scope_10/inner/tuning/M2/search_table.csv     M2   1.00000  10.000000           q0.25 FAILED Adaptive maximum iterations reached
scopes/scope_14/inner/tuning/M2/search_table.csv     M2 100.00000 100.000000           q0.25 FAILED Adaptive maximum iterations reach